## Create disorder-order benchmark dataset
Import data dump from MoMaP. Restrict dataset to motif classes which are mapped to ELM classes. Select an equal number of clusters from Q2, Q3, and Q4 of the dataset by number of structures within each ELM class. Create structure files with the chains involved in the interaction listed in the dataset. Save dataset and structure files as the disorder-order benchmark dataset.

In [ ]:
import json
import numpy as np
import pandas as pd
#import requests
#from tqdm import tqdm

# Read PDB instances file - contains information about motif and domain within the structure, as well as identifiers for specificity classes and pockets
with open('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/MoMaP_data-dump_20240605/PDB_instances_more_details.json', encoding='utf8') as json_file:
    motifdict = json.load(json_file)

In [2]:
print(motifdict[0])
print(len(motifdict))
print(max([len(x['interaction_pdb']) for x in motifdict]))
# Each item contains only 1 PDB - makes flattening into dataframe fairly simple

{'pmid': '-1', 'source': 'PDB', 'curator_username': 'curation_robot', 'motif_instance_id': None, 'curation_id': 220100, 'version_id': 223652, 'previous_version_id': None, 'is_active_version': True, 'date_created': '2022-02-02T19:09:32.730421', 'elm_accession': None, 'motif_start_residue_number': 323, 'motif_end_residue_number': 354, 'motif_sequence': 'LDGEYFTLQIRGRERFEMFRELNEALELKDAQ', 'motif_protein_uniprot_accession': 'P04637', 'domain_start_residue_number': 342, 'domain_end_residue_number': 354, 'domain_protein_uniprot_accession': 'P04637', 'domain_pfam_accession': 'PF07710', 'regex': None, 'curated_specificity_acc': None, 'pfam_names': 'P53_tetramer', 'pfam_long_names': 'P53 tetramerisation motif', 'clan_name': '', 'clan_id': '', 'interaction_type': 'extended helical binding region', 'curation_required': [{'field': 'pmid', 'param': '-1', 'details': "Couldn't find paper."}], 'interaction_kd': {'value': None, 'min': None, 'max': None, 'sd': None}, 'evidence_classes': None, 'evidence_

In [3]:
# Attach items within nested dictionaries/arrays to the parent dictionary
# Remove nested items
for entry in motifdict:
    for keys in entry['other_notes']:
        if keys != 'disordered interface footprint chains' and keys != 'domain matches':
            entry[keys] = entry['other_notes'][keys]
    for keys2 in entry['interaction_kd']:
        entry[keys2] = entry['interaction_kd'][keys2]
    entry['pdb_id'] = entry['interaction_pdb'][0]
    entry.pop('other_notes')
    entry.pop('interaction_kd')
    entry.pop('interaction_pdb')
    entry.pop('curation_required')
print(motifdict[0])

{'pmid': '-1', 'source': 'PDB', 'curator_username': 'curation_robot', 'motif_instance_id': None, 'curation_id': 220100, 'version_id': 223652, 'previous_version_id': None, 'is_active_version': True, 'date_created': '2022-02-02T19:09:32.730421', 'elm_accession': None, 'motif_start_residue_number': 323, 'motif_end_residue_number': 354, 'motif_sequence': 'LDGEYFTLQIRGRERFEMFRELNEALELKDAQ', 'motif_protein_uniprot_accession': 'P04637', 'domain_start_residue_number': 342, 'domain_end_residue_number': 354, 'domain_protein_uniprot_accession': 'P04637', 'domain_pfam_accession': 'PF07710', 'regex': None, 'curated_specificity_acc': None, 'pfam_names': 'P53_tetramer', 'pfam_long_names': 'P53 tetramerisation motif', 'clan_name': '', 'clan_id': '', 'interaction_type': 'extended helical binding region', 'evidence_classes': None, 'evidence_logic': None, 'curator_instance_assessment': 'true positive', 'curator_reliability_assessment': 'certain', 'conditional_motif': None, 'conditional_motif_data': None,

In [4]:
motifmaster = pd.DataFrame(motifdict)
# Drop any rows without PDB ID or chain annotations, as we are unable to extract the correct interfaces for these observations
motifmaster.dropna(subset=['motif chain','domain chain','pdb_id'], inplace=True)
print(motifmaster.info())
print(motifmaster.head())

<class 'pandas.core.frame.DataFrame'>
Index: 8060 entries, 0 to 8067
Data columns (total 65 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   pmid                                                    8060 non-null   object 
 1   source                                                  8060 non-null   object 
 2   curator_username                                        8060 non-null   object 
 3   motif_instance_id                                       0 non-null      object 
 4   curation_id                                             8060 non-null   int64  
 5   version_id                                              8060 non-null   int64  
 6   previous_version_id                                     0 non-null      float64
 7   is_active_version                                       8060 non-null   bool   
 8   date_created                               

In [5]:
# Filter for only SLiMs
motifmaster.query("interaction_type == 'SLiM'", inplace=True)
print(motifmaster.shape[0])
# Check whether any column provides a unique identifier
print(len(motifmaster['pmid'].unique()))
print(len(motifmaster['specificity_instance_id'].unique()))
print(len(motifmaster['pdb_id'].unique()))
# None are, so we will create a new identifier based on unique combinations of PDB ID, motif chain, and domain chain
motifmaster['uniqueStruct'] = pd.factorize(motifmaster['motif chain'] + motifmaster['domain chain'] + motifmaster['pdb_id'])[0]
# No duplicates - successfully found a unique way to identify each observation
print(motifmaster.loc[np.where(motifmaster.uniqueStruct.duplicated())])

7023
1856
2066
3956
Empty DataFrame
Columns: [pmid, source, curator_username, motif_instance_id, curation_id, version_id, previous_version_id, is_active_version, date_created, elm_accession, motif_start_residue_number, motif_end_residue_number, motif_sequence, motif_protein_uniprot_accession, domain_start_residue_number, domain_end_residue_number, domain_protein_uniprot_accession, domain_pfam_accession, regex, curated_specificity_acc, pfam_names, pfam_long_names, clan_name, clan_id, interaction_type, evidence_classes, evidence_logic, curator_instance_assessment, curator_reliability_assessment, conditional_motif, conditional_motif_data, tags, update, pocket_acc, specificity_acc, specificity_instance_id, evidence_methods, interface_disorder_proportion, multipartite_interface, disordered interface footprint total, construct_uniprot_similarity, fragment_description, mapped_to_uniprot, polymer_details, polymer_length, polymer_mutations, polymer_type, motif chain, secondary structure, second

In [ ]:
# Read in mapping file between specificity accessions and ELM classes
with open('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/MoMaP_data-dump_20240605/MoMaP-ELM_mapping.json', encoding='utf8') as json_file:
    elmmapping = json.load(json_file)

In [7]:
# Investigate number of ELM classes per specificity class
print(sum([len(x['motif_classes']) for x in elmmapping if x['motif_classes'] is not None])/len(elmmapping)) # Avg num. ELM classes per specificity class
print(max([len(x['motif_classes']) for x in elmmapping if x['motif_classes'] is not None]))
print(len(elmmapping))
elmmapping = [x for x in elmmapping if x['motif_classes'] is not None]
print(len(elmmapping)) # Number of mappings after removing mappings with no ELM classes
elmmapping = [x for x in elmmapping if len(x['motif_classes']) == 1]
print(len(elmmapping)) # Number of mappings after removing multiplicative mappings (specificity -> ELM direction)
# Multiple ELM -> specificity mappings is not an issue, and would actually help with putative clustering

# 40 specificity classes with multiple ELM mappings
# 119 specificity classes with no ELM mappings

0.835
6
400
281
241


In [8]:
# Attach items within nested motif classes dictionary to the parent dictionary
# Some other items are arrays, but these do not need to be processed yet and will be left as arrays
# Remove nested motif classes dictionary
for entry in elmmapping:
    for keys in entry['motif_classes'][0]:
        entry[keys] = entry['motif_classes'][0][keys]
    entry.pop('motif_classes')
print(elmmapping[0])

{'specificity_id': '14-3-3_phosphopocket', 'specificity_acc': 'IDIC000001', 'specificity_name': '14-3-3 phosphopeptide motif', 'specificity_functional_class': ['Binding'], 'specificity_termini_binding': None, 'specificity_prototype_structure': '3MHR', 'specificity_consensus': 'R[^DE]{0,2}[^DEPG]([ST])(([FWYLMV].)|([^PRIKGN]P)|([^PRIKGN].{2,4}[VILMFWYP]))', 'specificity_taxonomic_range_present': ['Eukaryota'], 'specificity_taxonomic_range_missing': None, 'pocket_domain_ids': ['PF00244'], 'pocket_name': '14-3-3 phosphopeptide-binding pocket', 'pocket_prototype_structure': '3P1N', 'no_of_instances': 157, 'no_of_sources': 8, 'no_of_curators': 8, 'no_of_papers': 137, 'no_of_domain_proteins': 20, 'no_of_domain_pfams': 2, 'no_of_motif_proteins': 124, 'last_modified': '2024-05-14T07:17:00.800316', 'date_created': '2020-11-13T13:23:24.007153', 'subtypes': [{'id': 307, 'name': '1a', 'consensus': 'R[^DEPG]([ST])[FWYLMV].', 'specificity_acc': 'IDIC000001'}, {'id': 308, 'name': '1b', 'consensus': '

In [9]:
elmmapdf = pd.DataFrame(elmmapping)
print(len(elmmapdf.specificity_acc.unique()))
print(len(elmmapdf.elm_accession.unique()))
# Specificity classes and ELM classes currently map 1-1
print(elmmapdf.info())
print(elmmapdf.head())


241
241
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 27 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   specificity_id                       241 non-null    object
 1   specificity_acc                      241 non-null    object
 2   specificity_name                     241 non-null    object
 3   specificity_functional_class         241 non-null    object
 4   specificity_termini_binding          30 non-null     object
 5   specificity_prototype_structure      174 non-null    object
 6   specificity_consensus                241 non-null    object
 7   specificity_taxonomic_range_present  240 non-null    object
 8   specificity_taxonomic_range_missing  4 non-null      object
 9   pocket_domain_ids                    232 non-null    object
 10  pocket_name                          234 non-null    object
 11  pocket_prototype_structure           

In [10]:
# Drop ELM accession column from initial dataframe - will merge new one
motifmaster = motifmaster.drop(columns='elm_accession', axis=1)
# Merge initial dataframe and specificity - ELM mapping
final = pd.merge(left=motifmaster, right=elmmapdf, on='specificity_acc', how='left')
# Create unique identifier for each interface
final['IFID'] = ['IF400'+str(x) for x in final.uniqueStruct]
#print(final.info())
# 7023 structures, 4533 with ELM accession
# Drop observations without ELM accession - will stick with motif classes with more thorough documentation
final.dropna(subset='elm_accession', axis=0, inplace=True)
print(final.info())

<class 'pandas.core.frame.DataFrame'>
Index: 4533 entries, 1 to 7021
Data columns (total 92 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   pmid                                                    4533 non-null   object 
 1   source                                                  4533 non-null   object 
 2   curator_username                                        4533 non-null   object 
 3   motif_instance_id                                       0 non-null      object 
 4   curation_id                                             4533 non-null   int64  
 5   version_id                                              4533 non-null   int64  
 6   previous_version_id                                     0 non-null      float64
 7   is_active_version                                       4533 non-null   bool   
 8   date_created_x                             

In [36]:
## After initial benchmarks, instituted a minimum length policy for motifs (or any interacting chain)
# Any chain must not have fewer than 4 interacting residues
# This information is not available in the dataset, and attempts to filter the dataset before making interface size restrictions have led to benchmark sets with poor composition (i.e. clusters are reduced to 1,2,3 structures or are eliminated completely)
# Will instead calculate interface size for every structure and filter the whole dataset
# Files will get saved in the process but will get deleted after

# RUN ON 16.08.2024 AND 19.08.2024

#import re
#from Bio import PDB
#from tqdm import tqdm
#from pdb_file_processing import get_two_chains
#from pdb_file_processing import get_interface_residues
#filepath = '/home/stromjoe/DMI'

#approved_ifs = []

#for i in tqdm(range(0,7023)):
    ## Obtain PDB file
    #name = final['pdb_id'][i].lower()
    #try:
        #url = f'https://files.rcsb.org/view/{name}.pdb'
        #f = open(f'{filepath}/{name}.pdb', 'w')
        #ext = 'pdb'
        #if re.search("404 Not Found", requests.get(url).text):
            #url = f'https://files.rcsb.org/view/{name}.cif'
            #f = open(f'{filepath}/{name}.cif', 'w')
            #ext = 'cif' 
    #except:
        #try:
            #url = f'https://files.rcsb.org/view/{name}.cif'
            #f = open(f'{filepath}/{name}.cif', 'w')
            #ext = 'cif'
        #except:
            #print('Error in obtaining'+name)
    #f.write(requests.get(url).text)
    #f.close()

    ## Extract two chains
    #chains = [final['motif chain'][i],final['domain chain'][i]]
    #get_two_chains(name, filepath, ext, chains, name_to_save=f"{name}_{final['IFID'][i]}")
    #min_len = get_interface_residues(name=f"{name}_{final['IFID'][i]}", filepath=filepath, ext=ext)
    #if min_len >= 4:
        #approved_ifs.append(f"{name}_{final['IFID'][i]}")

100%|██████████| 160/160 [05:32<00:00,  2.08s/it]


In [ ]:
#with open('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/approved_dmi_ifs.txt','a') as f:
    #for item in approved_ifs:
        #f.write(str(item)+',')

In [ ]:
with open('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/approved_dmi_ifs.txt','r') as f:
    contents = f.readlines()
    approved_ifs = contents[0].split(',')
approved_ifs = [x.split('_')[-1] for x in approved_ifs]    
print(approved_ifs)
print(len(approved_ifs))
final = final.query("IFID in @approved_ifs")

['IF4000', 'IF4001', 'IF4003', 'IF4004', 'IF4005', 'IF4006', 'IF4007', 'IF4008', 'IF4009', 'IF40010', 'IF40011', 'IF40012', 'IF40013', 'IF40014', 'IF40015', 'IF40016', 'IF40017', 'IF40018', 'IF40021', 'IF40022', 'IF40023', 'IF40024', 'IF40025', 'IF40026', 'IF40027', 'IF40028', 'IF40029', 'IF40030', 'IF40031', 'IF40032', 'IF40033', 'IF40034', 'IF40035', 'IF40036', 'IF40037', 'IF40038', 'IF40039', 'IF40040', 'IF40041', 'IF40042', 'IF40043', 'IF40044', 'IF40045', 'IF40046', 'IF40047', 'IF40048', 'IF40049', 'IF40050', 'IF40051', 'IF40052', 'IF40053', 'IF40054', 'IF40055', 'IF40056', 'IF40057', 'IF40058', 'IF40059', 'IF40060', 'IF40062', 'IF40064', 'IF40065', 'IF40066', 'IF40067', 'IF40068', 'IF40069', 'IF40070', 'IF40071', 'IF40072', 'IF40073', 'IF40074', 'IF40075', 'IF40076', 'IF40077', 'IF40078', 'IF40079', 'IF40080', 'IF40081', 'IF40082', 'IF40083', 'IF40084', 'IF40085', 'IF40086', 'IF40087', 'IF40088', 'IF40089', 'IF40090', 'IF40091', 'IF40092', 'IF40093', 'IF40094', 'IF40095', 'IF4009

In [12]:
# Number of structures with annotated DMI
print('Number of structures: ', final.shape[0])
# Number of specificity classes
print('Number of specificity classes: ', len(final.specificity_acc.unique()))
# Number of pocket classes
print('Number of pocket classes: ', len(final.pocket_acc.unique()))
# Statistics for number of domains per pocket class
def num_unique_pfam(x):
    y = len(x['domain_pfam_accession'].unique())
    return y
print('Statistics for number of domains per pocket class:\n',
      final.groupby('pocket_acc').apply(num_unique_pfam, include_groups=False).describe())
# Statistics for number of specificity classes per pocket
def num_unique_spec(x):
    y = len(x['specificity_acc'].unique())
    return y
print('Statistics for number of specificity classes per pocket:\n', 
      final.groupby('pocket_acc').apply(num_unique_spec, include_groups=False).describe())
# Statistics for number of structures per specificity class
print('Statistics for number of structures per specificity class:\n',
      final.groupby('specificity_acc')['uniqueStruct'].count().describe())
# Statistics for number of structures per pocket
print('Statistics for number of structures per pocket:\n',
      final.groupby('pocket_acc')['uniqueStruct'].count().describe())

Number of structures:  4072
Number of specificity classes:  169
Number of pocket classes:  132
Statistics for number of domains per pocket class:
 count    131.000000
mean       1.412214
std        0.849165
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        5.000000
dtype: float64
Statistics for number of specificity classes per pocket:
 count    131.000000
mean       1.274809
std        0.936881
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        7.000000
dtype: float64
Statistics for number of structures per specificity class:
 count     169.000000
mean       24.094675
std       102.411997
min         1.000000
25%         2.000000
50%         6.000000
75%        15.000000
max      1216.000000
Name: uniqueStruct, dtype: float64
Statistics for number of structures per pocket:
 count     131.000000
mean       30.946565
std       123.105603
min         1.000000
25%         2.000000
50%         6.000000
75%    

In [13]:
spec_accs = final.groupby('specificity_acc', as_index=False)['uniqueStruct'].count().sort_values('uniqueStruct', ascending=False)
spec_accs.reset_index(inplace=True, drop=True)
print(spec_accs)
spec_accs = spec_accs.iloc[0:126,]
import random
q4 = spec_accs.iloc[0:42,].sample(n=8, axis='index', random_state=18181818)
q3 = spec_accs.iloc[42:84,].sample(n=20, axis='index', random_state=36363636)
q2 = spec_accs.iloc[84:,].sample(n=30, axis='index', random_state=54545454)
spec_accs_sample = pd.concat([q4,q3,q2], axis=0)
spec_accs_sample.uniqueStruct.sum() #Total number of structures among all sampled specificity classes

    specificity_acc  uniqueStruct
0        IDIC000119          1216
1        IDIC000001           452
2        IDIC000131           283
3        IDIC000170           175
4        IDIC000110           100
..              ...           ...
164      IDIC000126             1
165      IDIC000135             1
166      IDIC000139             1
167      IDIC000164             1
168      IDIC000378             1

[169 rows x 2 columns]


np.int64(778)

In [ ]:
accs_to_fetch = list(spec_accs_sample.specificity_acc)
sampled_final = final.query('specificity_acc in @accs_to_fetch')
print(sampled_final.shape[0])
print(sampled_final.groupby('specificity_acc').pmid.count())
print(sampled_final.groupby('specificity_acc').pmid.count().describe())

# Generate a factor variable for the clusters, here based on motif specificity classes
sampled_final['trueClusters'] = pd.factorize(sampled_final.specificity_acc)[0]

# Rename or reformat some variables for input into later functions
sampled_final['pdb_id'] = [x.lower() for x in sampled_final.pdb_id]
sampled_final['fullID'] = sampled_final[['pdb_id', 'IFID']].agg('-'.join, axis=1)
sampled_final['pairID'] = sampled_final.pocket_acc
sampled_final['ClusterID'] = sampled_final.elm_identifier

sampled_final.sort_values(by='ClusterID', inplace=True)

sampled_final.to_csv('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/dmi_clusters.csv', index=False)

778
specificity_acc
IDIC000002      8
IDIC000008      3
IDIC000009      4
IDIC000012      2
IDIC000031      4
IDIC000046      9
IDIC000048     11
IDIC000050      6
IDIC000054      4
IDIC000056      4
IDIC000064     16
IDIC000065      6
IDIC000066      7
IDIC000067      3
IDIC000072     10
IDIC000073      2
IDIC000077     48
IDIC000084      3
IDIC000101      6
IDIC000103      8
IDIC000104      3
IDIC000110    100
IDIC000111      2
IDIC000143      7
IDIC000145     14
IDIC000147      4
IDIC000149      3
IDIC000152      5
IDIC000154      6
IDIC000159      4
IDIC000160      3
IDIC000161     15
IDIC000170    175
IDIC000173      5
IDIC000176     11
IDIC000188      8
IDIC000190      6
IDIC000209      4
IDIC000225      4
IDIC000226     20
IDIC000230      4
IDIC000238      3
IDIC000246     55
IDIC000256     10
IDIC000258     19
IDIC000264     56
IDIC000265      3
IDIC000266      3
IDIC000299      2
IDIC000305      6
IDIC000307      6
IDIC000308      3
IDIC000315     11
IDIC000318      8
IDIC0003

C:\Users\stromjoe\AppData\Local\Temp\ipykernel_19572\2743336281.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sampled_final['trueClusters'] = pd.factorize(sampled_final.specificity_acc)[0]
C:\Users\stromjoe\AppData\Local\Temp\ipykernel_19572\2743336281.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sampled_final['pdb_id'] = [x.lower() for x in sampled_final.pdb_id]
C:\Users\stromjoe\AppData\Local\Temp\ipykernel_19572\2743336281.py:12: SettingWithCopyWarning: 
A value is trying to be set on a cop

In [ ]:
# Download identified PDB files and extract two desired chains for DMI_v2 dataset
# RUN ON 30.10.2024

#from Bio import PDB
#from tqdm import tqdm
#from pdb_file_processing import get_two_chains
#filepath = '/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/DMI'

#for i,r in tqdm(sampled_final.iterrows()):
    ## Obtain PDB file
    #name = r['pdb_id'].lower()
    #try:
        #url = f'https://files.rcsb.org/view/{name}.pdb'
        #f = open(f'{filepath}/{name}.pdb', 'w')
        #ext = 'pdb'
    #except:
        #try:
            #url = f'https://files.rcsb.org/view/{name}.cif'
            #f = open(f'{filepath}/{name}.cif', 'w')
            #ext = 'cif'
        #except:
            #print('Error in obtaining'+name)
    #f.write(requests.get(url).text)
    #f.close()

    ## Extract two chains
    #chains = [r['motif chain'],r['domain chain']]
    #get_two_chains(name, filepath, ext, chains, name_to_save=f"{name}-{r['IFID']}")

778it [12:01,  1.08it/s]
